## DSPy Prompts to Assess Readability in Questions

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
from dspy.evaluate.evaluate import Evaluate
from dspy.teleprompt import COPRO, MIPRO, BootstrapFewShot, BootstrapFewShotWithRandomSearch
import json
import tiktoken

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
# create gpt-4 model
gpt4 = dspy.OpenAI(model='gpt-4-turbo', max_tokens=500, api_key=open_ai_api_key)  

gpt4("which openai model are you? Are you gpt4?")

["Yes, I am based on OpenAI's GPT-4 model. How can I assist you today?"]

In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=500, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

In [4]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.encoding_for_model(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [5]:
constants = {
    "question_json_format": """{
        "cell_type": "question",
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }"""
}

In [6]:
def parse_json_str(json_str):
    return json.loads(json_str)

def fill_in_constants(input_str):
    for key in constants:
        input_str = input_str.replace("{"+key+"}", constants[key])
    return input_str

In [7]:
test_inputs = [
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you think military security is more important and should have more budget allocation than social security?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "medium"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you agree or disagree with the following statement? The social discrepancies in Germany will certainly continue to exist.",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "medium"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you agree or disagree with the following statement? The social differences in Germany will certainly continue to exist.",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "high"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you think the increase in the rate of immigration, controlling for the economy, is higher or lower than the increase in the rate of crime in your area?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "medium"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you favor or oppose requiring states to have 60% of the approval of voters to raise state taxes?",
            "response_categories": [
                {"id": 0, "text": "Favor"},
                {"id": 1, "text": "Oppose"}
            ],
        },
        "readability": "high"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you favor or oppose not allowing the state to raise state taxes without approval of 60% of voters?",
            "response_categories": [
                {"id": 0, "text": "Favor"},
                {"id": 1, "text": "Oppose"}
            ],
        },
        "readability": "medium"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Should your local government not raise taxes? Should not the government never have offered abortion license?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "low"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Has the externality of market deregulation been taken care of?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "low"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Did the government handle negative impacts of ending food subsidy well?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "high"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Do you support or oppose tort reform?",
            "response_categories": [
                {"id": 0, "text": "Support"},
                {"id": 1, "text": "Oppose"}
            ],
        },
        "readability": "low"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Should people held on terror related crimes have the right of habeas corpus?",
            "response_categories": [
                {"id": 0, "text": "Yes"},
                {"id": 1, "text": "No"}
            ],
        },
        "readability": "low"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "Are you very likely, somewhat likely, somewhat unlikely, or very unlikely to hire a tax preparer next year?",
            "response_categories": [
                {"id": 0, "text": "Very unlikely"},
                {"id": 1, "text": "Somewhat unlikely"},
                {"id": 2, "text": "Somewhat likely"},
                {"id": 3, "text": "Very likely"}
            ],
        },
        "readability": "medium"
    },
    {
        "question":{
            "cell_type": "question",
            "response_format": "closed",
            "description": "",
            "main_text": "How likely or unlikely are you to hire a tax preparer next year?",
            "response_categories": [
                {"id": 0, "text": "Very unlikely"},
                {"id": 1, "text": "Somewhat unlikely"},
                {"id": 2, "text": "Somewhat likely"},
                {"id": 3, "text": "Very likely"}
            ],
        },
        "readability": "high"
    },
]

In [8]:
# helper function to test out a program on question bank

def test_program(program, test_inputs, verbose=True):

    num_correct = 0

    # rand_int = random.randint(1, 100)

    # running the predictor
    for test_input in test_inputs:
        # stringify the input
        test_input_str = json.dumps(test_input["question"])
        # test_input_str = test_input["main_text"]
        # result = program(question=test_input_str, config=dict(temperature=0.7+0.0001*rand_int))
        result = program(question=test_input_str, reading_level="third grade")
        rationale = result.rationale
        output = result.readability
        if verbose:
            print(f"The readability of the question '{test_input["question"]["main_text"]}' is {output}.") 
            print(f"The expected readability is {test_input["readability"]}.")
            print(f"Rationale: {rationale}")
            print()
            print()

        expected_output = test_input["readability"]
        if expected_output in output.lower():
            num_correct += 1

    return num_correct

### Create AssessReadability Signature

In [30]:
# Create a class-based DSPy Signature to assess readability of a question

sig_desc = """Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives."""

input_desc = """{"question": "The question to classify. The input will be a JSON with the following structure: {question_json_format}",
"reading_level": "The reading level to assess the question against."}"""

output_desc = """{"readability": "The readability of the question. Only output one of the following strings: low, medium, or high. Don't include any other information in the output."}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_desc)
output_descriptions_json = parse_json_str(output_desc)

class AssessReadability(dspy.Signature):
    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    reading_level = dspy.InputField(desc=fill_in_constants(input_descriptions_json["reading_level"]))
    readability = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["readability"]))

# set the signature description
AssessReadability.__doc__ = sig_desc

print(AssessReadability.__doc__)

Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives.


#### Test AssessReadability Signature

In [21]:
# test out AssessReadability

readability = dspy.ChainOfThought(AssessReadability)

num_correct1 = test_program(readability, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")


Number of correct predictions: 5/13. 38.46153846153847% accuracy.


In [22]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives.

---

Follow the following format.

Question: The questi

### Create AssessReadabilityModule

In [31]:
# Create a module from AssessReadability

class AssessReadabilityModule(dspy.Module):
    
    def __init__(self):

        super().__init__()
        
        self.readability = dspy.ChainOfThought(AssessReadability)

    def forward(self, question, reading_level, temp=0.7):

        # this needs to return a dict and not a string for Evaluate to work
        return self.readability(question=question, reading_level=reading_level,
                                config=dict(temperature=temp))

### Optimize AssessReadabilityModule

#### Load data and create metric

In [32]:
filename = "readability_outputs"

# Load the training data
with open(f"generated_questions/{filename}_train.json", 'r') as f:
    train_data = json.load(f)    

# iterate through the data and construct a DSPy Example
trainset_desc = []
for item in train_data:
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    trainset_desc.append(dspy.Example(question=question_str, readability=item["readability"], reading_level="third grade").with_inputs("question", "reading_level"))   

# iterate through the data and construct a DSPy Example
trainset_no_desc = []
for item in train_data:
    # set the description field to an empty string
    item["question"]["description"] = ""
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    trainset_no_desc.append(dspy.Example(question=question_str, readability=item["readability"], reading_level="third grade").with_inputs("question", "reading_level"))  

# Load the validation data
with open(f"generated_questions/{filename}_val.json", 'r') as f:
    val_data = json.load(f)

# iterate through the data and construct a DSPy Example
valset = []
for item in val_data:
    # set the description field to an empty string
    item["question"]["description"] = ""
    # stringify the input
    question_str = json.dumps(item["question"])
    # question_str = item["question"]["main_text"]
    valset.append(dspy.Example(question=question_str, readability=item["readability"], reading_level="third grade").with_inputs("question", "reading_level"))   

print(len(trainset_desc), len(trainset_no_desc), len(valset))
print(valset[0].question)
print(trainset_desc[0].question)
print(trainset_no_desc[0].question)

285 285 75
{"response_format": "closed", "description": "", "main_text": "How is the satisfaction with public transportation described by you?", "response_categories": [{"id": 1, "text": "Very satisfied"}, {"id": 2, "text": "Somewhat satisfied"}, {"id": 3, "text": "Neutral"}, {"id": 4, "text": "Somewhat dissatisfied"}, {"id": 5, "text": "Very dissatisfied"}]}
{"response_format": "open", "description": "Gathering opinions on public transportation.", "main_text": "By the community, how is the public transportation system preferred to be enhanced?", "response_categories": []}
{"response_format": "open", "description": "", "main_text": "By the community, how is the public transportation system preferred to be enhanced?", "response_categories": []}


In [33]:
# Create a metric
def validate_readability(example, pred, trace=None):

    # make sure rationale is above a certain length
    rationale_length = len(pred.rationale.split(" "))

    # print(pred.rationale)

    if pred.readability.lower() not in ["low", "medium", "high"]:
        # print(f"Invalid prediction: {pred.readability}")
        # another way of finding the category
        pred_category = ""
        if "low" in pred.readability.lower():
            pred_category = "low"
        elif "medium" in pred.readability.lower():
            pred_category = "medium"
        elif "high" in pred.readability.lower():
            pred_category = "high"
        else:
            return False
        
        return (example.readability.lower() == pred_category) and (rationale_length > 10)

    return (example.readability.lower() == pred.readability.lower()) and (rationale_length > 10)

#### Non optimized program

In [34]:
non_optimized_program = AssessReadabilityModule()

In [35]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(non_optimized_program, metric=validate_readability)

  0%|          | 0/75 [00:00<?, ?it/s]

Average Metric: 40 / 75  (53.3): 100%|██████████| 75/75 [00:18<00:00,  3.97it/s]

Average Metric: 40 / 75  (53.3%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' 'False' 'False' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(


,question,example_readability,reading_level,rationale,pred_readability,validate_readability
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How is the satisfaction with public transportation described by you?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",medium,third grade,"produce the readability. We will first check if the question is concise and in active voice. Then, we will ensure that it does not contain...",high,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are you not dissatisfied with the local government's response times?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"":...",medium,third grade,"produce the readability. We first look at the question structure, which contains a negative phrasing (""Are you not dissatisfied""). This adds complexity and may confuse...",low,False
2,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How frequently do you utilize the PM2.5 AQI to assess outdoor activity safety?"", ""response_categories"": [{""id"": 1, ""text"": ""Daily""}, {""id"": 2,...",low,third grade,"assess the readability. We first consider the vocabulary used in the question. ""Utilize,"" ""PM2.5 AQI,"" and ""outdoor activity safety"" may be too complex for a...",medium,False
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are the public services not meeting your needs?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"": 3, ""text"":...",medium,third grade,"produce the readability. We first consider that the question contains a negative phrase ""not meeting your needs"", which might be confusing for a third-grade reading...",low,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Do you find the regulations surrounding the city's recycling initiatives to be overly convoluted for the average resident to comprehend...",low,third grade,"produce the readability. We first need to consider the vocabulary and sentence structure used in the question. Next, we should assess if any complex concepts...",low,✔️ [True]


53.33

In [36]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives.

---

Follow the following format.

Question: The questi

In [35]:
# get the number of tokens
test_output = """produce the readability. We see that the question is simple and straightforward, asking about the brightness of street lights at night, which is a common experience for most people. The response options are clear and easy to understand as well.

Readability: high"""

num_tokens = num_tokens_from_string(test_output, "gpt-3.5-turbo")

print(f"Number of tokens: {num_tokens}")

Number of tokens: 49


In [37]:
# test on question bank

num_correct1 = test_program(non_optimized_program, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

Number of correct predictions: 4/13. 30.76923076923077% accuracy.


#### BootstrapFewShot

In [48]:
# set up optimizer (repeat 10 times)
config = dict(max_bootstrapped_demos=3, max_labeled_demos=5, max_rounds=2, max_errors=5)
# max_bootstrapped_demos: Refers to the maximum number of demonstrations that will be bootstrapped. Bootstrapping in this context likely means generating new training examples based on the predictions of a teacher module or some other process
# max_labeled_demos: Refers to the maximum number of labeled demonstrations (examples) that will be used for training the student module directly

fewshot_optimizer = BootstrapFewShot(metric=validate_readability, **config)
fewshot_optimized_program = fewshot_optimizer.compile(AssessReadabilityModule(), trainset=trainset_desc)

  0%|          | 0/285 [00:00<?, ?it/s]

Bootstrapped 3 full traces after 1 examples in round 1.


In [49]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(fewshot_optimized_program, metric=validate_readability)

  0%|          | 0/75 [00:00<?, ?it/s]

Average Metric: 48 / 75  (64.0): 100%|██████████| 75/75 [00:14<00:00,  5.06it/s]

Average Metric: 48 / 75  (64.0%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' 'False' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(


,question,example_readability,reading_level,rationale,pred_readability,validate_readability
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How is the satisfaction with public transportation described by you?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",medium,third grade,"produce the readability. The question contains a grammatical error (""described by you"" instead of ""do you describe""), making it less clear and concise for a...",medium,✔️ [True]
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are you not dissatisfied with the local government's response times?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"":...",medium,third grade,"produce the readability. The question contains a double negative (""not dissatisfied""), which can be confusing for a third-grade reader.",low,False
2,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How frequently do you utilize the PM2.5 AQI to assess outdoor activity safety?"", ""response_categories"": [{""id"": 1, ""text"": ""Daily""}, {""id"": 2,...",low,third grade,"produce the readability. The question contains acronyms (PM2.5 AQI) that are not defined, making it difficult for a third-grade reader to understand.",low,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are the public services not meeting your needs?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"": 3, ""text"":...",medium,third grade,produce the readability. The question contains a double negative which can be confusing for a third-grade reader.,low,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Do you find the regulations surrounding the city's recycling initiatives to be overly convoluted for the average resident to comprehend...",low,third grade,"produce the readability. We see that the question contains complex structures, jargon, and uses passive voice, making it difficult for a third-grade reader to understand.",low,✔️ [True]


64.0

In [50]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives.

---

Follow the following format.

Question: The questi

In [51]:
# test on question bank

num_correct1 = test_program(fewshot_optimized_program, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

Number of correct predictions: 7/13. 53.84615384615385% accuracy.


In [65]:
# Let's save the optimized programs
fewshot_optimized_program.save('compiled_modules/assess_readability_few_shot_desc2.json')

#### BootstrapFewShotWithRandomSearch

In [53]:
# max_bootstrapped_demos: Refers to the maximum number of demonstrations that will be bootstrapped. Bootstrapping in this context likely means generating new training examples based on the predictions of a teacher module or some other process
# max_labeled_demos: Refers to the maximum number of labeled demonstrations (examples) that will be used for training the student module directly
fewshot_search_optimizer = BootstrapFewShotWithRandomSearch(metric=validate_readability, max_bootstrapped_demos=2, num_candidate_programs=8, num_threads=64)
fewshot_search_optimized_program = fewshot_search_optimizer.compile(AssessReadabilityModule(), trainset=trainset_desc, valset=valset)

Going to sample between 1 and 2 traces per predictor.
Will attempt to train 8 candidate sets.


Average Metric: 6 / 11  (54.5):  13%|█▎        | 10/75 [00:00<00:00, 1666.92it/s]

Average Metric: 40 / 75  (53.3): 100%|██████████| 75/75 [00:00<00:00, 1703.54it/s]

Average Metric: 40 / 75  (53.3%)
Score: 53.33 for set: [0]
New best score: 53.33 for seed -3
Scores so far: [53.33]
Best score: 53.33



Average Metric: 44 / 75  (58.7): 100%|██████████| 75/75 [00:02<00:00, 35.12it/s]


...

Average Metric: 48 / 75  (64.0%)
Score: 64.0 for set: [16]
Scores so far: [53.33, 58.67, 64.0, 68.0, 54.67, 52.0, 62.67, 64.0, 66.67, 64.0]
Best score: 68.0
Average of max per entry across top 1 scores: 0.68
Average of max per entry across top 2 scores: 0.92
Average of max per entry across top 3 scores: 0.9866666666666667
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0


  1%|▏         | 4/285 [00:03<04:27,  1.05it/s]


Bootstrapped 2 full traces after 5 examples in round 0.


Average Metric: 45 / 75  (60.0): 100%|██████████| 75/75 [00:03<00:00, 24.58it/s]

Average Metric: 45 / 75  (60.0%)
Score: 60.0 for set: [16]
Scores so far: [53.33, 58.67, 64.0, 68.0, 54.67, 52.0, 62.67, 64.0, 66.67, 64.0, 60.0]
Best score: 68.0
Average of max per entry across top 1 scores: 0.68
Average of max per entry across top 2 scores: 0.92
Average of max per entry across top 3 scores: 0.9866666666666667
Average of max per entry across top 5 scores: 1.0
Average of max per entry across top 8 scores: 1.0
Average of max per entry across top 9999 scores: 1.0
11 candidate programs found.


In [54]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(fewshot_search_optimized_program, metric=validate_readability)

Average Metric: 1 / 2  (50.0):   1%|▏         | 1/75 [00:00<?, ?it/s] 

Average Metric: 51 / 75  (68.0): 100%|██████████| 75/75 [00:00<00:00, 1784.78it/s]

Average Metric: 51 / 75  (68.0%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' 'False' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(


,question,example_readability,reading_level,rationale,pred_readability,validate_readability
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How is the satisfaction with public transportation described by you?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",medium,third grade,Readability: low,low,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are you not dissatisfied with the local government's response times?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"":...",medium,third grade,"produce the readability. We need to identify any negatives or double negatives, check for clarity in the question structure, and ensure it is concise and...",low,False
2,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How frequently do you utilize the PM2.5 AQI to assess outdoor activity safety?"", ""response_categories"": [{""id"": 1, ""text"": ""Daily""}, {""id"": 2,...",low,third grade,"produce the readability. We need to assess the question for clarity, simplicity, and lack of complex structures that may hinder understanding.",low,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are the public services not meeting your needs?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"": 3, ""text"":...",medium,third grade,"produce the readability. We need to assess the question for simplicity, clarity, and lack of double negatives.",low,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Do you find the regulations surrounding the city's recycling initiatives to be overly convoluted for the average resident to comprehend...",low,third grade,"produce the readability. We need to consider the complexity of the sentence structure, the presence of jargon or acronyms, the use of negatives, and the...",low,✔️ [True]


68.0

In [55]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives.

---

Follow the following format.

Question: The questi

In [56]:
# test on question bank

num_correct1 = test_program(fewshot_search_optimized_program, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

Number of correct predictions: 4/13. 30.76923076923077% accuracy.


In [66]:
# Let's save the optimized programs
fewshot_search_optimized_program.save('compiled_modules/assess_readability_few_shot_search_desc2.json')

#### COPRP with GPT3.5 `prompt_model`

In [38]:
copro_teleprompter1 = COPRO(
    prompt_model=gpt3_turbo,
    metric=validate_readability,
    verbose=True,
)

kwargs = dict(num_threads=64, display_progress=True, display_table=0) # Used in Evaluate class in the optimization process

copro_optimized_program1 = copro_teleprompter1.compile(AssessReadabilityModule(), trainset=trainset_desc, eval_kwargs=kwargs)





You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be afraid to be creative.

---

Follow the following format.

Basic Instruction: The initial instructions before optimization
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Basic Instruction: Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable: (1) Question should meet the inputted reading level. (2) Question should not contain basic spelling or grammar mistakes. (3) Question should be concise. (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or

Average Metric: 100 / 285  (35.1): 100%|██████████| 285/285 [00:07<00:00, 40.43it/s]


Average Metric: 100 / 285  (35.1%)




Focus on sentence structure, vocabulary choice, and overall clarity in the question. Ensure that the question is written in a straightforward and clear manner, following basic grammar rules and avoiding complex or jargon-heavy language. Be concise and make sure that the question is easy to understand by a wide audience.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Reading Level: The reading level to assess the question against.

Reasoning: Let's think step by step in order to ${produce the readability}. We ...

Improving Question Readability The readability of the question. Only output one of the following strings: low, medium, or high. 

Average Metric: 153 / 285  (53.7): 100%|██████████| 285/285 [00:08<00:00, 31.83it/s]


Average Metric: 153 / 285  (53.7%)




Consider the information density, clarity, and conciseness of a question to determine its readability. Check for appropriate reading level, absence of spelling and grammar errors, clarity in terms of potential jargon usage, defined acronyms, clarification of proper nouns, active voice, simplicity in grammar, and the avoidance of negatives/double negatives.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Reading Level: The reading level to assess the question against.

Reasoning: Let's think step by step in order to ${produce the readability}. We ...

Checklist for question readability The readability of the question. Only output one of the 

...

Average Metric: 79 / 285  (27.7%)




Instruction #11: To optimize language model performance, apart from regularly upgradation just like punishment rehabilitation.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Reading Level: The reading level to assess the question against.

Reasoning: Let's think step by step in order to ${produce the readability}. We ...

Optimzation task code The readability of the question. Only output one of the following strings: low, medium, or high. Don't include any other information in the output.

---

Question: {"response_format": "closed", "description": "Question about community event participation", "main_text": "Have you attended a community e

Average Metric: 106 / 285  (37.2): 100%|██████████| 285/285 [00:08<00:00, 32.86it/s]


Average Metric: 106 / 285  (37.2%)




Honing in on question readability necessitates considering a question's reading level, grammar, spelling, conciseness, integration of jargon-free language with proper definitions and minimization of unnurtured acronyms, descriptiveness concerning proper nouns, emphatic employment of active constructs, inhibition in over-propositioning and over-reliance on logical operators, and holistic elimination of negative or doubled negative usages. Fine-tuning this multidimensional evaluation probe can optimally ascertain the epitome of question readability, thus paving the path for heightened efficiency and effectiveness in varied language models.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an 

Average Metric: 89 / 285  (31.2): 100%|██████████| 285/285 [00:05<00:00, 48.63it/s]

Average Metric: 89 / 285  (31.2%)




The instructions are already quite detailed and comprehensive in guiding the language model to enhance question readability. However, to further improve the model's performance, we can consider incorporating a step that involves analyzing the context in which the questions are presented. This context could include the overall theme, purpose, audience, or setting of the questions, which can impact the ideal level of readability needed. Adapting the readability criteria to fit the specific context of the questions could potentially lead to even more tailored and effective evaluations.

Proposed Instruction: "In addition to the readability criteria specified, analyze and adapt the evaluation for question readability based on the contextual factors such as theme, purpose, audience, and setting. Ensure that the level of readability aligns with these contextual elements to optimize the comprehension and relevance of the questions.

---

Follow the follow

In [39]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(copro_optimized_program1, metric=validate_readability)

  0%|          | 0/75 [00:00<?, ?it/s]

Average Metric: 36 / 75  (48.0): 100%|██████████| 75/75 [00:20<00:00,  3.67it/s]

Average Metric: 36 / 75  (48.0%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' 'False' 'False' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(


,question,example_readability,reading_level,rationale,pred_readability,validate_readability
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How is the satisfaction with public transportation described by you?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",medium,third grade,"produce the readability. We will ensure that the question is clear, concise, and free of complex language or terms that may be difficult for a...",: high,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are you not dissatisfied with the local government's response times?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"":...",medium,third grade,"produce the readability. We first need to identify any complex or convoluted language used in the question, such as the use of double negatives (""not...",: low,False
2,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How frequently do you utilize the PM2.5 AQI to assess outdoor activity safety?"", ""response_categories"": [{""id"": 1, ""text"": ""Daily""}, {""id"": 2,...",low,third grade,"produce the readability. We will simplify complex terms like ""PM2.5 AQI"" to make it easier to understand for a third-grade reading level. We will also...",: high,False
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are the public services not meeting your needs?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"": 3, ""text"":...",medium,third grade,"produce the readability. We see that the question uses the logical operator ""not"" and poses a negative statement. This can potentially confuse readers, especially those...",: low,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Do you find the regulations surrounding the city's recycling initiatives to be overly convoluted for the average resident to comprehend...",low,third grade,"produce the readability. We will simplify complex terms, avoid jargon, and use clear and straightforward language to ensure comprehension by a third-grade reading level audience.",: low,✔️ [True]


48.0

In [40]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





New Instruction: When evaluating question readability, ensure that it aligns with the provided reading level, maintains proper spelling and grammar, is concise, devoid of jargon and undefined acronyms, adequately explains proper nouns, uses active voice, minimizes logical operators, and avoids negations or double negatives.

---

Follow the following format.

Question: The question to classify. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Reading Level: The reading level to assess the question against.

Reasoning: Let's think step by step in order to ${produce the readability}. We ...

Overall Question Quality The readability of the question. Only output one of the following strings: low, medium, or high. Don't include any other information i

In [41]:
# test on question bank

num_correct1 = test_program(copro_optimized_program1, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

Number of correct predictions: 7/13. 53.84615384615385% accuracy.


In [42]:
# Let's save the optimized programs
copro_optimized_program1.save('compiled_modules/assess_readability_copro_optimized1.json')

#### COPRP with GPT 4 `prompt_model`

In [43]:
copro_teleprompter = COPRO(
    prompt_model=gpt4,
    metric=validate_readability,
    verbose=True,
)

kwargs = dict(num_threads=64, display_progress=True, display_table=0) # Used in Evaluate class in the optimization process

copro_optimized_program2 = copro_teleprompter.compile(AssessReadabilityModule(), trainset=trainset_desc, eval_kwargs=kwargs)





You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be afraid to be creative.

---

Follow the following format.

Basic Instruction: The initial instructions before optimization
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Basic Instruction: Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable: (1) Question should meet the inputted reading level. (2) Question should not contain basic spelling or grammar mistakes. (3) Question should be concise. (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or

Average Metric: 161 / 285  (56.5): 100%|██████████| 285/285 [00:00<00:00, 2053.86it/s]


Average Metric: 161 / 285  (56.5%)




You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be afraid to be creative.

---

Follow the following format.

Basic Instruction: The initial instructions before optimization
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Basic Instruction: Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable: (1) Question should meet the inputted reading level. (2) Question should not contain basic spelling or grammar mistakes. (3) Question should be concise. (4) Question should not contain potential jargon (e.g. special words or expressions that are

Average Metric: 169 / 285  (59.3): 100%|██████████| 285/285 [00:00<00:00, 1675.71it/s]


Average Metric: 169 / 285  (59.3%)




You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be afraid to be creative.

---

Follow the following format.

Basic Instruction: The initial instructions before optimization
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Basic Instruction: Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable: (1) Question should meet the inputted reading level. (2) Question should not contain basic spelling or grammar mistakes. (3) Question should be concise. (4) Question should not contain potential jargon (e.g. special words or expressions that are

...

Average Metric: 154 / 285  (54.0%)




You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.

Your task is to propose a new instruction that will lead a good language model to perform the task even better. Don't be afraid to be creative.

---

Follow the following format.

Attempted Instructions: ${attempted_instructions}
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Attempted Instructions:
[1] «Instruction #1: Basic Instruction: Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable: (1) Question should meet the inputted reading level. (2) Questi

Average Metric: 81 / 285  (28.4): 100%|██████████| 285/285 [00:00<00:00, 1904.51it/s]


Average Metric: 81 / 285  (28.4%)




You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.

Your task is to propose a new instruction that will lead a good language model to perform the task even better. Don't be afraid to be creative.

---

Follow the following format.

Attempted Instructions: ${attempted_instructions}
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Attempted Instructions:
[1] «Instruction #1: Basic Instruction: Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable: (1) Question should meet the inputted reading level. (2) Questio

Average Metric: 111 / 285  (38.9): 100%|██████████| 285/285 [00:00<00:00, 1694.87it/s]

Average Metric: 111 / 285  (38.9%)




You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.

Your task is to propose a new instruction that will lead a good language model to perform the task even better. Don't be afraid to be creative.

---

Follow the following format.

Attempted Instructions: ${attempted_instructions}
Proposed Instruction: The improved instructions for the language model
Proposed Prefix For Output Field: The string at the end of the prompt, which will help the model start solving the task

---

Attempted Instructions:
[1] «Instruction #1: Enhance your evaluation of a question's readability, classifying it as high, medium, or low, using refined criteria which directly support comprehendibility suitable for the expected reading audience. Multiple poi

In [44]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(copro_optimized_program2, metric=validate_readability)

Average Metric: 31 / 64  (48.4):  84%|████████▍ | 63/75 [00:00<00:00, 1852.98it/s]

Average Metric: 38 / 75  (50.7): 100%|██████████| 75/75 [00:00<00:00, 1875.06it/s]

Average Metric: 38 / 75  (50.7%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['False' 'False' 'False' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(


,question,example_readability,reading_level,rationale,pred_readability,validate_readability
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How is the satisfaction with public transportation described by you?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",medium,third grade,produce the readability. We first analyze the structure and wording of the question to ensure it aligns with a third-grade reading level. We check for...,low,False
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are you not dissatisfied with the local government's response times?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"":...",medium,third grade,"produce the readability. We will analyze the question for the presence of negatives or double negatives, complexity of language, and clarity in conveying the intended...",low,False
2,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How frequently do you utilize the PM2.5 AQI to assess outdoor activity safety?"", ""response_categories"": [{""id"": 1, ""text"": ""Daily""}, {""id"": 2,...",low,third grade,"assess the readability. We need to ensure that the question is simple and straightforward for a third-grade reading level. We will check for clear language,...",high,False
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are the public services not meeting your needs?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"": 3, ""text"":...",medium,third grade,"produce the readability. We will first check if the question aligns with a third-grade reading level, then look for any spelling or grammar errors, assess...",low,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Do you find the regulations surrounding the city's recycling initiatives to be overly convoluted for the average resident to comprehend...",low,third grade,produce the readability. We need to consider if the vocabulary and sentence structure are suitable for a third-grade reading level. We should also check for...,low,✔️ [True]


50.67

In [45]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Basic Instruction: Assess the readability of a question using the criteria: readability level alignment, absence of spelling/grammar errors, conciseness, clear absence of jargon, defined acronyms, described proper nouns, active voice expression, minimalistic usage of propositions/logical operators, and avoidance of negatives or double negatives. Categorize readability as high, medium, or low.

Proposed Instruction: Evaluate the readability level of a text by assigning a category of high, medium, or low readability based on specific criteria. Review whether the text achieves clarity and simplicity suitable for its intent by verifying: 1) Proper anatomy of questions matching the stated reading level without contradicting it; 2) Flawless use of spelling and grammar; 3) Brevity and directness in phraseology; 4) Omission of specialized jargon or strictly define any that appears; 5) Clear definition for every acronym mentioned; 6) Sufficient explanation of referenced proper nouns such as

In [46]:
# test on question bank

num_correct1 = test_program(copro_optimized_program2, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

Number of correct predictions: 6/13. 46.15384615384615% accuracy.


In [19]:
# Let's save the optimized programs
copro_optimized_program2.save('compiled_modules/assess_readability_copro_optimized2.json')

#### MIPRO

In [58]:
# defaults from MIPRO class
# num_candidates = num_new_prompts_generated
mipro_teleprompter = MIPRO(prompt_model=gpt4, task_model=gpt3_turbo, metric=validate_readability, num_candidates=10, init_temperature=1.0)

# defaults from MIPRO class
kwargs = dict(num_threads=64, display_progress=True, display_table=0)

# defaults from MIPRO class
mipro_optimized_program = mipro_teleprompter.compile(AssessReadabilityModule(), trainset=trainset_desc, num_trials=100, max_bootstrapped_demos=3, max_labeled_demos=5, eval_kwargs=kwargs)


Please be advised that based on the parameters you have set, the maximum number of LM calls is projected as follows:

- Task Model: 285 examples in dev set * 100 trials * # of LM calls in your program = (28500 * # of LM calls in your program) task model calls
- Prompt Model: # data summarizer calls (max 10) + 10 * 1 lm calls in program = 20 prompt model calls

Estimated Cost Calculation:

Total Cost = (Number of calls to task model * (Avg Input Token Length per Call * Task Model Price per Input Token + Avg Output Token Length per Call * Task Model Price per Output Token) 
            + (Number of calls to prompt model * (Avg Input Token Length per Call * Task Prompt Price per Input Token + Avg Output Token Length per Call * Prompt Model Price per Output Token).

For a preliminary estimate of potential costs, we recommend you perform your own calculations based on the task
and prompt models you intend to use. If the projected costs exceed your budget or expectations, you may consider:


  2%|▏         | 5/285 [00:04<04:16,  1.09it/s]


Bootstrapped 3 full traces after 6 examples in round 0.


  2%|▏         | 7/285 [00:06<04:06,  1.13it/s]


Bootstrapped 3 full traces after 8 examples in round 0.


...

Average Metric: 77 / 100  (77.0%)


Average Metric: 74 / 100  (74.0): 100%|██████████| 100/100 [00:00<00:00, 1470.24it/s]


Average Metric: 74 / 100  (74.0%)


Average Metric: 69 / 85  (81.2): 100%|██████████| 85/85 [00:00<00:00, 1308.85it/s]
[I 2024-04-15 11:13:22,776] Trial 99 finished with value: 77.1940350877193 and parameters: {'3166422625040_predictor_instruction': 0, '3166422625040_predictor_demos': 7}. Best is trial 25 with value: 77.1940350877193.


Average Metric: 69 / 85  (81.2%)
Returning readability = ChainOfThought(AssessReadability(question, reading_level -> readability
    instructions='Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:\n        (1) Question should meet the inputted reading level.\n        (2) Question should not contain basic spelling or grammar mistakes.\n        (3) Question should be concise.\n        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.\n        (5) Question should not contain any acronyms that are not defined.\n        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.\n        (7) Question should be in active voice.\n        (8) Question should have as few propositions and logi

In [64]:
# Evaluate the optimized program

evaluate_program = Evaluate(devset=valset, num_threads=5, display_progress=True, display_table=5)

evaluate_program(mipro_optimized_program, metric=validate_readability)

Average Metric: 14 / 21  (66.7):  27%|██▋       | 20/75 [00:00<00:00, 1667.48it/s]

Average Metric: 50 / 75  (66.7): 100%|██████████| 75/75 [00:00<00:00, 1932.89it/s]

Average Metric: 50 / 75  (66.7%)



c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\dspy\evaluate\evaluate.py:266: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['✔️ [True]' 'False' '✔️ [True]' 'False' '✔️ [True]']' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[:, metric_name] = df[metric_name].apply(


,question,example_readability,reading_level,rationale,pred_readability,validate_readability
0,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How is the satisfaction with public transportation described by you?"", ""response_categories"": [{""id"": 1, ""text"": ""Very satisfied""}, {""id"": 2, ""text"": ""Somewhat...",medium,third grade,"produce the readability. The question is concise and in active voice, but it contains a grammatical error with ""described by you"" instead of ""do you...",medium,✔️ [True]
1,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are you not dissatisfied with the local government's response times?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"":...",medium,third grade,"produce the readability. The question contains a double negative with ""not dissatisfied,"" which can be confusing for a third-grade reading level.",low,False
2,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""How frequently do you utilize the PM2.5 AQI to assess outdoor activity safety?"", ""response_categories"": [{""id"": 1, ""text"": ""Daily""}, {""id"": 2,...",low,third grade,"produce the readability. The question contains the acronym ""AQI"" without defining it, which could be confusing for a third-grade reading level.",low,✔️ [True]
3,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Are the public services not meeting your needs?"", ""response_categories"": [{""id"": 1, ""text"": ""Yes""}, {""id"": 2, ""text"": ""No""}, {""id"": 3, ""text"":...",medium,third grade,"produce the readability. The question contains a negative in the form of ""not meeting your needs."" This could be confusing for a third-grade reading level.",low,False
4,"{""response_format"": ""closed"", ""description"": """", ""main_text"": ""Do you find the regulations surrounding the city's recycling initiatives to be overly convoluted for the average resident to comprehend...",low,third grade,"produce the readability. The question is quite verbose and contains complex language like ""overly convoluted"" and ""comprehend fully"". This may be challenging for a third-grade...",low,✔️ [True]


66.67

In [60]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Assess the readability of a question to high, medium, or low. Keep in mind the following qualities, for a question to be readable:
        (1) Question should meet the inputted reading level.
        (2) Question should not contain basic spelling or grammar mistakes.
        (3) Question should be concise.
        (4) Question should not contain potential jargon (e.g. special words or expressions that are used by a particular profession or group and are difficult for others to understand) that are not defined.
        (5) Question should not contain any acronyms that are not defined.
        (6) Question should not mention proper nouns (e.g. names of specific people, places, or organizations) without describing what they are.
        (7) Question should be in active voice.
        (8) Question should have as few propositions and logical operators as possible.
        (9) Question should not have negatives or double negatives.

---

Follow the following format.

Question: The questi

In [61]:
# test on question bank

num_correct1 = test_program(mipro_optimized_program, test_inputs, verbose=False)

print(f"Number of correct predictions: {num_correct1}/{len(test_inputs)}. {num_correct1/len(test_inputs)*100}% accuracy.")

Number of correct predictions: 4/13. 30.76923076923077% accuracy.


In [63]:
# Let's save the optimized programs
mipro_optimized_program.save('compiled_modules/assess_readability_mipro_optimized.json')